# Logistic Regression Behavioral Fraud Risk Assessment
Comparison model for multiclass behavioral fraud-risk assessment.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize


## Load Data


In [ ]:
train_file = "behavior_fraud_train_featured.csv"
test_file = "behavior_fraud_test_featured.csv"

train = pd.read_csv(train_file)
test = pd.read_csv(test_file)

print("Training data:", train.shape)
print("Testing data :", test.shape)
display(train.head())


## Prepare Risk Labels


In [ ]:
risk_map = {
    "Low": "Low Risk", "Moderate": "Moderate Risk",
    "High": "High Risk", "Critical": "Critical Risk",
    "Low Risk": "Low Risk", "Moderate Risk": "Moderate Risk",
    "High Risk": "High Risk", "Critical Risk": "Critical Risk"
}

train["risk_level"] = train["risk_level"].astype(str).str.strip().replace(risk_map)
test["risk_level"] = test["risk_level"].astype(str).str.strip().replace(risk_map)

risk_order = ["Low Risk", "Moderate Risk", "High Risk", "Critical Risk"]
y_train = train["risk_level"].copy()
y_test = test["risk_level"].copy()

unexpected = set(y_train.unique()).union(y_test.unique()) - set(risk_order)
if unexpected:
    raise ValueError(f"Unexpected risk labels: {sorted(unexpected)}")

display(y_train.value_counts().reindex(risk_order, fill_value=0).to_frame("Count"))


## Risk-Class Distribution


In [ ]:
distribution = pd.DataFrame({
    "Train": y_train.value_counts().reindex(risk_order, fill_value=0),
    "Test": y_test.value_counts().reindex(risk_order, fill_value=0)
})
display(distribution)
distribution.plot(kind="bar", figsize=(9,5))
plt.title("Risk-Class Distribution")
plt.xlabel("Risk Level"); plt.ylabel("Transactions"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()


## Behavioral Features


In [ ]:
leakage_columns = ["risk_level", "is_fraud", "risk_score", "risk_reasons"]
identifier_columns = ["transaction_id", "customer_id", "account_id", "TransactionID", "CustomerID"]
datetime_columns = ["transaction_date", "transaction_datetime", "transaction_time", "date", "datetime", "timestamp"]
drop_columns = leakage_columns + identifier_columns + datetime_columns

X_train = train.drop(columns=[c for c in drop_columns if c in train.columns], errors="ignore").copy()
X_test = test.drop(columns=[c for c in drop_columns if c in test.columns], errors="ignore").copy()

for col in list(X_train.columns):
    name = col.lower()
    if any(term in name for term in ["date", "time", "datetime", "timestamp"]) and name not in {"transaction_hour", "is_night", "unusual_time"}:
        X_train.drop(columns=col, inplace=True, errors="ignore")
        X_test.drop(columns=col, inplace=True, errors="ignore")

features = [c for c in X_train.columns if c in X_test.columns]
X_train = X_train[features]
X_test = X_test[features]

print("Behavioral features used:", len(features))
display(pd.DataFrame({"Feature": features}))

## Preprocessing and Scaling


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool", "str"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## Logistic Regression Model


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1500,
    class_weight="balanced",
    solver="lbfgs",
    random_state=42
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", lr_model)
])

model


## Train Model


In [ ]:
model.fit(X_train, y_train)
print("Training completed.")


## Predictions


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

preview = pd.DataFrame({
    "Actual Risk": y_test.reset_index(drop=True),
    "Predicted Risk": pd.Series(y_pred).reset_index(drop=True)
})
display(preview.head(10))


## Model Metrics


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
rows = [("Accuracy", accuracy)]
for avg in ["macro", "micro", "weighted"]:
    rows.extend([
        (f"{avg.title()} Precision", precision_score(y_test, y_pred, average=avg, zero_division=0)),
        (f"{avg.title()} Recall", recall_score(y_test, y_pred, average=avg, zero_division=0)),
        (f"{avg.title()} F1", f1_score(y_test, y_pred, average=avg, zero_division=0))
    ])

class_order = model.named_steps["classifier"].classes_
y_test_bin = label_binarize(y_test, classes=class_order)
try:
    macro_auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    weighted_auc = roc_auc_score(y_test_bin, y_proba, average="weighted", multi_class="ovr")
    micro_auc = roc_auc_score(y_test_bin, y_proba, average="micro", multi_class="ovr")
    rows.extend([("Macro ROC-AUC (OvR)", macro_auc), ("Micro ROC-AUC (OvR)", micro_auc), ("Weighted ROC-AUC (OvR)", weighted_auc)])
except ValueError as e:
    print("ROC-AUC could not be calculated:", e)

metrics_df = pd.DataFrame(rows, columns=["Metric", "Score"])
print("Logistic Regression Model Performance")
display(metrics_df.round(4))
metrics_df.to_csv("logistic_regression_behavioral_fraud_risk_enhanced_metrics.csv", index=False)


## Classification Report


In [ ]:
report_df = pd.DataFrame(
    classification_report(
        y_test, y_pred, labels=risk_order,
        output_dict=True, zero_division=0
    )
).T

display(report_df.round(4))
report_df.to_csv("logistic_regression_classification_report.csv")


## Metrics Graph


In [ ]:
metrics_df.set_index("Metric").plot(kind="bar", legend=False, figsize=(9,5))
plt.title("Logistic Regression Performance Metrics")
plt.ylabel("Score"); plt.ylim(0,1); plt.xticks(rotation=25)
plt.tight_layout(); plt.show()


## Class-Wise Performance


In [ ]:
class_metrics = report_df.loc[risk_order, ["precision", "recall", "f1-score"]]
display(class_metrics.round(4))
class_metrics.plot(kind="bar", figsize=(10,5))
plt.title("Precision, Recall and F1 by Risk Class")
plt.ylabel("Score"); plt.ylim(0,1); plt.xticks(rotation=0)
plt.legend(["Precision", "Recall", "F1"])
plt.tight_layout(); plt.show()


## Confusion Matrix


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=risk_order, display_labels=risk_order,
    cmap="Blues", values_format="d", ax=ax
)
plt.title("Logistic Regression Confusion Matrix")
plt.xticks(rotation=25); plt.tight_layout(); plt.show()


## Normalized Confusion Matrix


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=risk_order, display_labels=risk_order,
    normalize="true", cmap="Blues", values_format=".2f", ax=ax
)
plt.title("Normalized Confusion Matrix")
plt.xticks(rotation=25); plt.tight_layout(); plt.show()


## Actual vs Predicted Risk


In [ ]:
risk_comparison = pd.DataFrame({
    "Actual": y_test.value_counts().reindex(risk_order, fill_value=0),
    "Predicted": pd.Series(y_pred).value_counts().reindex(risk_order, fill_value=0)
})
display(risk_comparison)
risk_comparison.plot(kind="bar", figsize=(9,5))
plt.title("Actual vs Predicted Risk Levels")
plt.ylabel("Transactions"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()


## Probability-Based Predictions


In [ ]:
# Align probability columns with the classifier's class order
class_order = model.named_steps["classifier"].classes_
probability_df = pd.DataFrame(
    y_proba,
    columns=[f"P({risk})" for risk in class_order]
)

probability_df["Actual Risk"] = y_test.reset_index(drop=True)
probability_df["Predicted Risk"] = pd.Series(y_pred).reset_index(drop=True)
probability_df["Confidence"] = y_proba.max(axis=1)

display(probability_df.head(15).round(4))
probability_df.to_csv("logistic_regression_risk_predictions.csv", index=False)


## Prediction Confidence


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(probability_df["Confidence"], bins=20, edgecolor="black")
plt.title("Prediction Confidence")
plt.xlabel("Highest Predicted Probability"); plt.ylabel("Transactions")
plt.tight_layout(); plt.show()


## Most Influential Features


In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_

mean_influence = np.mean(np.abs(coefficients), axis=0)

influence_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean Absolute Coefficient": mean_influence
}).sort_values("Mean Absolute Coefficient", ascending=False)

top_features = influence_df.head(20)
display(top_features.round(4))

plt.figure(figsize=(10,7))
plt.barh(
    top_features["Feature"][::-1],
    top_features["Mean Absolute Coefficient"][::-1]
)
plt.title("Top 20 Influential Behavioral Features")
plt.xlabel("Mean Absolute Coefficient")
plt.tight_layout(); plt.show()

influence_df.to_csv("logistic_regression_feature_influence.csv", index=False)


## Metric Comparison
Accuracy and multiclass precision, recall and F1 scores.


In [ ]:
plot_metrics = metrics_df[~metrics_df["Metric"].str.contains("ROC-AUC")].copy()
plt.figure(figsize=(12,6))
plt.bar(plot_metrics["Metric"], plot_metrics["Score"])
plt.ylim(0,1.05)
plt.ylabel("Score")
plt.title("Logistic Regression - Classification Metrics")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()


## Class-wise Metrics Heatmap
Precision, recall and F1-score for each risk level.


In [ ]:
class_perf = report_df.loc[risk_order, ["precision", "recall", "f1-score"]]
fig, ax = plt.subplots(figsize=(8,5))
im = ax.imshow(class_perf.values, aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(3), ["Precision", "Recall", "F1-score"])
ax.set_yticks(range(len(risk_order)), risk_order)
for i in range(class_perf.shape[0]):
    for j in range(class_perf.shape[1]):
        ax.text(j, i, f"{class_perf.iloc[i,j]:.3f}", ha="center", va="center")
plt.colorbar(im, ax=ax, label="Score")
plt.title("Logistic Regression - Class-wise Performance")
plt.tight_layout(); plt.show()


## ROC Curves
One-vs-rest ROC curves for the four risk classes.


In [ ]:
plt.figure(figsize=(9,7))
for i, cls in enumerate(class_order):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    class_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC={class_auc:.3f})")
plt.plot([0,1], [0,1], "--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression - One-vs-Rest ROC Curves")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout(); plt.show()


## Error Rate by Risk Class


In [ ]:
error_df = pd.DataFrame({"Actual": y_test.reset_index(drop=True), "Predicted": pd.Series(y_pred)})
error_df["Incorrect"] = error_df["Actual"] != error_df["Predicted"]
error_rate = error_df.groupby("Actual")["Incorrect"].mean().reindex(risk_order)
plt.figure(figsize=(9,5))
plt.bar(error_rate.index, error_rate.values)
plt.ylim(0, max(1.0, float(error_rate.max()) * 1.1))
plt.ylabel("Misclassification Rate")
plt.title("Logistic Regression - Error Rate by Risk Class")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()
display((error_rate * 100).round(2).rename("Error %").to_frame())


## Correct vs Incorrect Predictions by Risk Class


In [ ]:
ct = pd.crosstab(error_df["Actual"], np.where(error_df["Incorrect"], "Incorrect", "Correct")).reindex(risk_order, fill_value=0)
ct.plot(kind="bar", figsize=(10,5))
plt.title("Logistic Regression - Correct vs Incorrect Predictions")
plt.xlabel("Actual Risk Level"); plt.ylabel("Transactions")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## Metric Notes
- **Accuracy:** overall correct predictions.
- **Precision:** how often a predicted class is correct.
- **Recall:** how many actual cases of a class are detected.
- **F1-score:** balance between precision and recall.
- **Macro average:** gives every risk class equal importance.
- **Weighted average:** accounts for class frequency.
- **ROC-AUC:** measures how well the model separates each class from the others.

For this multiclass fraud-risk problem, macro F1 and macro recall are useful alongside accuracy because performance on the less frequent high-risk classes should not be hidden by the larger classes.
